In [12]:
# Install required packages (run once)
!pip install -q datasets tiktoken

In [13]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import random
import pickle
import math

device = (
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)

# Hyperparameters — scaled up for OpenWebText on Kaggle T4 (16GB VRAM)
batch_size = 32
block_size = 256          # longer context for real web text
max_iters = 3000
learning_rate = 3e-4
eval_iters = 100
eval_interval = 500
n_embd = 512
n_head = 8
n_layer = 8
dropout = 0.1

print(f"Device: {device}")

Device: cuda


In [14]:
import tiktoken

# Use GPT-2's BPE tokenizer — works perfectly with OpenWebText
enc = tiktoken.get_encoding('gpt2')
vocab_size = enc.n_vocab  # 50257

encode = lambda s: enc.encode(s, allowed_special={'<|endoftext|>'})
decode = lambda l: enc.decode(l)

print(f"Vocab size: {vocab_size:,}")

Vocab size: 50,257


In [15]:
from datasets import load_dataset

# Stream OpenWebText — avoids downloading 55GB upfront
# We'll tokenize and cache enough tokens for training
print("Loading OpenWebText (streaming)...")
dataset = load_dataset('Skylion007/openwebtext', split='train', streaming=True)
dataset = dataset.shuffle(seed=42, buffer_size=10_000)

# Collect ~50M tokens for training (manageable on Kaggle)
TARGET_TOKENS = 50_000_000
EOT = enc.encode('<|endoftext|>', allowed_special={'<|endoftext|>'})[0]  # document separator

all_tokens = []
for doc in dataset:
    tokens = encode(doc['text'])
    all_tokens.extend(tokens)
    all_tokens.append(EOT)  # separate documents
    if len(all_tokens) >= TARGET_TOKENS:
        break

print(f"Total tokens collected: {len(all_tokens):,}")

data = torch.tensor(all_tokens, dtype=torch.long)

# Train/val split (90/10)
n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]
print(f"Train: {len(train_data):,} tokens | Val: {len(val_data):,} tokens")

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

Loading OpenWebText (streaming)...


Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Total tokens collected: 50,000,276
Train: 45,000,248 tokens | Val: 5,000,028 tokens


In [16]:
class CausalSelfAttention(nn.Module):
    """Multi-head causal self-attention with fused QKV projection"""
    
    def __init__(self, n_embd, n_head):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        
        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = nn.Dropout(dropout)
        
        self.register_buffer('mask', torch.tril(torch.ones(block_size, block_size))
                             .view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.c_attn(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=dropout if self.training else 0.0)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.c_proj(out)
        out = self.dropout(out)
        return out


class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffwd = FeedForward(n_embd)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        
        # Weight tying
        self.token_embedding_table.weight = self.lm_head.weight
        
        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        B, T = index.shape
        tok_emb = self.token_embedding_table(index)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, index, max_new_tokens, temperature=0.8, top_k=40):
        for _ in range(max_new_tokens):
            index_cond = index[:, -block_size:]
            logits, _ = self.forward(index_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim=1)
        return index

model = GPTLanguageModel(vocab_size)
m = model.to(device)
print(f"Model parameters: {sum(p.numel() for p in m.parameters()):,}")

Model parameters: 51,066,368


In [17]:
# Optimizer with cosine LR schedule
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, betas=(0.9, 0.95), weight_decay=0.1)

def get_lr(it):
    warmup_iters = 200
    min_lr = learning_rate / 10
    if it < warmup_iters:
        return learning_rate * (it + 1) / warmup_iters
    decay_ratio = (it - warmup_iters) / (max_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

for iter in range(max_iters):
    lr = get_lr(iter)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter:5d} | lr: {lr:.2e} | train: {losses['train']:.4f} | val: {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

print(f"\nFinal loss: {loss.item():.4f}")
with open('model-owt.pkl', 'wb') as f:
    pickle.dump(model, f)
print('Model saved!')

step     0 | lr: 1.50e-06 | train: 10.9077 | val: 10.9070
step   500 | lr: 2.92e-04 | train: 6.1218 | val: 6.1369
step  1000 | lr: 2.49e-04 | train: 5.7530 | val: 5.7772
step  1500 | lr: 1.80e-04 | train: 5.5156 | val: 5.5548
step  2000 | lr: 1.06e-04 | train: 5.3796 | val: 5.4183
step  2500 | lr: 5.07e-05 | train: 5.3002 | val: 5.3399

Final loss: 5.1696
Model saved!


In [18]:
# Continue training from checkpoint
# Model is already in memory — just run more steps with a fresh LR schedule

more_iters = 10000
start_lr = 3e-5        # start low since we're already trained
min_lr = 3e-6

optimizer2 = torch.optim.AdamW(model.parameters(), lr=start_lr, betas=(0.9, 0.95), weight_decay=0.1)

def get_lr2(it):
    warmup_iters = 100
    if it < warmup_iters:
        return start_lr * (it + 1) / warmup_iters
    decay_ratio = (it - warmup_iters) / (more_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (start_lr - min_lr)

for iter in range(more_iters):
    lr = get_lr2(iter)
    for param_group in optimizer2.param_groups:
        param_group['lr'] = lr

    if iter % 500 == 0:
        losses = estimate_loss()
        print(f"step {iter:5d} | lr: {lr:.2e} | train: {losses['train']:.4f} | val: {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)

    optimizer2.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer2.step()
  
print(f"\nFinal loss: {loss.item():.4f}")
with open('model-owt-v2.pkl', 'wb') as f:
    pickle.dump(model, f)
print('Model saved as model-owt-v2.pkl!')

step     0 | lr: 3.00e-07 | train: 5.2719 | val: 5.3140
step   500 | lr: 2.99e-05 | train: 5.2353 | val: 5.2833
step  1000 | lr: 2.95e-05 | train: 5.2277 | val: 5.2823
step  1500 | lr: 2.87e-05 | train: 5.1988 | val: 5.2603
step  2000 | lr: 2.76e-05 | train: 5.1647 | val: 5.2592
step  2500 | lr: 2.63e-05 | train: 5.1698 | val: 5.2127
step  3000 | lr: 2.47e-05 | train: 5.1384 | val: 5.2117
step  3500 | lr: 2.29e-05 | train: 5.1116 | val: 5.2021
step  4000 | lr: 2.09e-05 | train: 5.1152 | val: 5.1925
step  4500 | lr: 1.88e-05 | train: 5.1189 | val: 5.1684
step  5000 | lr: 1.67e-05 | train: 5.0932 | val: 5.1572
step  5500 | lr: 1.46e-05 | train: 5.0806 | val: 5.1438
step  6000 | lr: 1.25e-05 | train: 5.0788 | val: 5.1518
step  6500 | lr: 1.05e-05 | train: 5.0971 | val: 5.1605
step  7000 | lr: 8.67e-06 | train: 5.0698 | val: 5.1320
step  7500 | lr: 7.03e-06 | train: 5.0682 | val: 5.1215
step  8000 | lr: 5.63e-06 | train: 5.0500 | val: 5.1197
step  8500 | lr: 4.50e-06 | train: 5.0508 | val:

In [19]:
prompt = 'The quick brown fox'
context = torch.tensor(encode(prompt), dtype=torch.long, device=device)
generated = decode(m.generate(context.unsqueeze(0), max_new_tokens=200, temperature=0.8, top_k=40)[0].tolist())
print(generated)

The quick brown fox in the world in the last three weeks. The only two are the highest-month-rounded and the most dangerous, a half-day, and the same-stowing. It’s like the original and full-time. It’s also a single-time way to get the way to do.

“The last time when I get around, it’s pretty much more. It’s too late, I won’t do it for me,” said T.

The second-round line will be one more than five more minutes after the next two weeks. The game will get from 5 days in this year, despite the second-time season with 20 yards.<|endoftext|>The second quarter, the first quarter of the season with three runs on the season of 5-2 in 2011, according to the first-round draft.

The season, with a few points in the last two-on-0-


In [20]:
num_params = sum(p.numel() for p in model.parameters())
num_trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {num_params:,}")
print(f"Trainable parameters: {num_trainable_params:,}")
print(f"Million parameters: {num_params / 1e6:.2f}M")

Total parameters: 51,066,368
Trainable parameters: 51,066,368
Million parameters: 51.07M
